# 07b · Pilot diagnostics from the saved results (no GPU)

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1 (pilot follow-up; reads only, no engines are built)

**Purpose.** Explain why the six pilot engines behave alike, using only the files notebook 07 already saved.

**In plain words.** Four questions: did every engine see its full calibration set; how many test videos sit close enough to the threshold to change decision; how far the scores moved between engines (not only the error counts); and in which part of the network the calibration scales changed.

**Expected inputs**
- Notebook 07 outputs: `engines/int8_entropy_<condition>_s<seed>/` (`engine_meta.json`, `scales.csv`) and `results/int8/<engine>/`; FP32 results `results/fp32/`; the ONNX model.

**Expected outputs**
- `QPAD_Project/results/int8/diagnostics/` (`engine_metadata_check.csv`, `threshold_margins.csv`, `score_shifts.csv`, `decision_flips.csv`, `scale_changes_by_tensor.csv`, `scale_changes_by_block.csv`)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
DRY_RUN_SEEDS = 1   # dry run: analyse seed 1 only (outputs go to _dry_run)
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb07b", paths.logs / "nb07b.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

In [ ]:
import pandas as pd
from qpad import diagnostics as D
from qpad.experiments import engine_tag
from qpad.pipeline import find_input
icfg = cfgs["int8"]
METHOD = icfg["calibration_method"]["primary"]
SEEDS = icfg["seeds"]["pilot"][:DRY_RUN_SEEDS] if DRY_RUN else icfg["seeds"]["pilot"]
CONDS = icfg["pilot_conditions"]
TAGS = {(c, s): engine_tag(METHOD, c, s) for s in SEEDS for c in CONDS}
# Always read the real pilot and FP32 results (dry-run FP32 scores come from an untrained model);
# DRY_RUN only limits the seeds and sends this notebook's own outputs to _dry_run.
ENGINE_DIRS = {t: find_input(paths, False, "engines", t) for t in TAGS.values()}
RESULT_DIRS = {t: find_input(paths, False, "results", f"int8/{t}") for t in TAGS.values()}
FP32_DIR = find_input(paths, False, "results", "fp32")
DIAG = OUT.results / "int8" / "diagnostics"
DIAG.mkdir(parents=True, exist_ok=True)
print("engines:", list(TAGS.values()))
print("writing to:", DIAG)

## 1. Did every engine see its full calibration set?
Each engine should have received 30 batches of 20 images (600) and written one scale per calibrated tensor. An engine with fewer batches did not see its whole calibration set.

In [ ]:
batches = icfg["calibration_budget"]["images"] // icfg["calibration_method"]["calibration_batch_size"]
META = D.engine_metadata_table(ENGINE_DIRS, batches)
META.to_csv(DIAG / "engine_metadata_check.csv", index=False)
display(META)
print("every engine saw its full calibration set:", bool(META["ok"].all()))

## 2. How many videos are close enough to the threshold to flip?
Only videos scoring near the threshold can change decision when INT8 shifts their score slightly. Counts per class of test videos within 0.02, 0.05 and 0.10 of each model's own threshold (policy B).

In [ ]:
fp32_videos, fp32_thr = D.read_test_scores(FP32_DIR)
rows = [D.margin_table(fp32_videos, fp32_thr).assign(model="FP32")]
for tag, d in RESULT_DIRS.items():
    videos, thr = D.read_test_scores(d, "B")
    rows.append(D.margin_table(videos, thr).assign(model=tag))
MARGINS = pd.concat(rows, ignore_index=True)
MARGINS = MARGINS[["model", *[c for c in MARGINS.columns if c != "model"]]]
MARGINS.to_csv(DIAG / "threshold_margins.csv", index=False)
display(MARGINS)

## 3. How far did the scores move, and which decisions flipped?
Continuous video-score changes are far more sensitive than error counts. Three kinds of comparison: INT8 natural vs FP32 (effect of quantization itself), unusual vs natural of the same seed (effect of the calibration mix) and natural seed 2 vs seed 1 (effect of the random video pick only).

In [ ]:
PAIRS = []
for s in SEEDS:
    natural = RESULT_DIRS[TAGS[("natural", s)]]
    PAIRS.append((f"INT8 natural s{s} vs FP32", FP32_DIR, natural))
    PAIRS += [(f"{c} vs natural, s{s}", natural, RESULT_DIRS[TAGS[(c, s)]]) for c in CONDS if c != "natural"]
if len(SEEDS) > 1:
    PAIRS.append((f"natural s{SEEDS[1]} vs natural s{SEEDS[0]} (seed only)",
                  RESULT_DIRS[TAGS[("natural", SEEDS[0])]], RESULT_DIRS[TAGS[("natural", SEEDS[1])]]))
SHIFTS, FLIPS_B = D.compare_pairs(PAIRS, "B")
_, FLIPS_A = D.compare_pairs(PAIRS, "A")
FLIPS = pd.concat([FLIPS_B, FLIPS_A], ignore_index=True)
SHIFTS.to_csv(DIAG / "score_shifts.csv", index=False)
FLIPS.to_csv(DIAG / "decision_flips.csv", index=False)
display(SHIFTS)
display(FLIPS)

## 4. Where in the network did the calibration scales change?
Each calibrated tensor is located in the exported graph (encoder block, pixel-map head) and its relative scale change against the natural engine of the same seed is summarised per block.

In [ ]:
ORDER = D.onnx_tensor_order(find_input(paths, False, "models", "deeppixbis_fp32.onnx"))
parts = []
for s in SEEDS:
    scales = {}
    for c in CONDS:
        tag = TAGS[(c, s)]
        table = pd.read_csv(ENGINE_DIRS[tag] / "scales.csv")
        scales[tag] = dict(zip(table["tensor"], table["scale"]))
    parts.append(D.scale_changes_by_layer(scales, TAGS[("natural", s)], ORDER))
CHANGES = pd.concat(parts, ignore_index=True)
BY_BLOCK = D.scale_change_by_block(CHANGES)
CHANGES.to_csv(DIAG / "scale_changes_by_tensor.csv", index=False)
BY_BLOCK.to_csv(DIAG / "scale_changes_by_block.csv", index=False)
print("calibrated tensors located in the ONNX graph:", int((CHANGES["block"] != "not in ONNX graph").sum()),
      "of", len(CHANGES))
display(BY_BLOCK)
display(CHANGES.sort_values("abs_rel", ascending=False).groupby("engine").head(10)
        [["engine", "tensor", "block", "op", "depth", "abs_rel"]])

In [ ]:
print("Engines with their full calibration set:", f"{int(META['ok'].sum())}/{len(META)}")
int8_margins = MARGINS[MARGINS["model"] != "FP32"]
print("Median number of test videos within 0.05 of the threshold, per class:",
      int8_margins.groupby("class")["within_0.05"].median().to_dict())
mix = FLIPS_B[FLIPS_B["comparison"].str.contains("vs natural,")]
print("Decision flips, unusual vs natural (policy B):")
print(mix.drop(columns=["policy"]).to_string(index=False))
print("Largest scale change per block (%):", BY_BLOCK.groupby("block")["max_change_pct"].max().round(1).to_dict())

**Next notebook:** `07c_pilot_diagnostics_gpu.ipynb`